# Sqoop:Postgres -> HDFS -> Hive (browse in Hue)

What this notebook does, in order:

1. **Create the source**: a `retail` schema in Postgres with 6 deliberately awkward tables and generated data (SQL files in `sql/`).
2. **Run Sqoop jobs** that pull those tables into HDFS: full loads, a free-form join query, an **append** incremental job and a **lastmodified + merge** incremental job.
3. **Register Hive tables** over the landed files (`sqoop_poc.stg_*`), then write curated Parquet copies (`sqoop_poc.*`).
4. **Validate** Postgres vs Hive (row counts, sums, null counts).
5. **Simulate a day of changes** in Postgres, rerun the incremental jobs, validate again.
6. Optional: a **Spark JDBC fallback** if Sqoop is not installed on the machine running the notebook.

Where to see the result in Hue: **File Browser** -> `/data/sqoop_poc/raw/` and the **Hive editor** -> database `sqoop_poc`.

The notebook machine needs: network access to Postgres, and the `sqoop`, `hdfs` commands on PATH (sections 3-5). Section 1 needs only Python.

## 0. Configuration (edit this cell)

In [ ]:
import os, sys
# ignore the old Spark 3.5.0 in /usr/local/spark
sys.path = [p for p in sys.path if not p.startswith("/usr/local/spark")]
import pyspark
os.environ["SPARK_HOME"] = os.path.dirname(pyspark.__file__)
print("PySpark", pyspark.__version__, "from", os.environ["SPARK_HOME"])

In [ ]:
import os, sys, subprocess, shlex, time

# ---- Postgres (source). Override with environment variables if you prefer.
from getpass import getpass

PG = dict(
    host="172.16.0.4",
    port=5432,
    dbname="sourcedb",
    user="postgres",
    password=os.getenv("PG_PASSWORD") or getpass("Postgres password: "),   # never commit a password
    connect_timeout=10,
)
PG_SCHEMA = "retail"
JDBC_URL = "jdbc:postgresql://{host}:{port}/{dbname}".format(**PG)



# ---- Hadoop / Hive (target)
HDFS_URI = "hdfs://172.16.0.4:9000"
HDFS_USER = os.getenv("HADOOP_USER_NAME", os.getenv("USER", "hadoop"))
RAW_BASE = "/data/sqoop_poc/raw"                       # where Sqoop lands the files
PW_FILE = "/user/%s/.pg_sqoop.pw" % HDFS_USER          # Sqoop password file (HDFS, mode 400)
HIVE_DB = "sqoop_poc"

# ---- Tools
SQL_DIR = "."
SQOOP_BIN = os.getenv("SQOOP_BIN", "sqoop")
HDFS_BIN = os.getenv("HDFS_BIN", "hdfs")

# True  -> use Sqoop (sections 3-5);  False -> skip them and use the Spark JDBC fallback (section 8)
USE_SQOOP = False

print("JDBC URL :", JDBC_URL)
print("HDFS raw :", HDFS_URI + RAW_BASE)

###### 1. Create the Postgres source (schema + data)

Tables and what makes each one tricky for an import:

| Table | Rows | Complexity |
|---|---|---|
| `branches` | 40 | small dimension, boolean, char(2) |
| `customers` | 10,000 | enum, JSONB, text[] array, timestamptz, nullable columns, notes with commas / quotes / newlines / tabs / unicode |
| `accounts` | 15,000 | FK to customers + branches, signed NUMERIC(18,2), nullable `close_date` / `credit_limit`, `updated_at` for lastmodified loads |
| `account_holders` | ~18,000 | **composite primary key** (many-to-many) |
| `transactions` | 200,000 | big table for parallel mappers, outliers and zero amounts, memo text with delimiters, `txn_id` for append loads |
| `fx_rates_log` | ~980 | **no primary key**, contains duplicate rows |

In [ ]:
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "psycopg2-binary"])
import psycopg2


def pg_connect(dbname=None, autocommit=False):
    cfg = dict(PG)
    if dbname:
        cfg["dbname"] = dbname
    conn = psycopg2.connect(**cfg)
    conn.autocommit = autocommit
    return conn


def pg_query(sql, params=None):
    conn = pg_connect()
    try:
        with conn.cursor() as cur:
            cur.execute(sql, params)
            return cur.fetchall()
    finally:
        conn.close()


def run_sql_file(name):
    with open(os.path.join(SQL_DIR, name), encoding="utf-8") as f:
        sql = f.read()
    conn = pg_connect()
    try:
        with conn.cursor() as cur:
            cur.execute(sql)
        conn.commit()
    finally:
        conn.close()
    print("ran", name)


# create the database if it does not exist
admin = pg_connect(dbname="postgres", autocommit=True)
with admin.cursor() as cur:
    cur.execute("SELECT 1 FROM pg_database WHERE datname = %s", (PG["dbname"],))
    if not cur.fetchone():
        cur.execute('CREATE DATABASE "%s"' % PG["dbname"])
        print("created database", PG["dbname"])
admin.close()

run_sql_file("01_schema.sql")
run_sql_file("02_data.sql")

In [ ]:
# Check what was generated, including the awkward data
checks = {
    "branches": "SELECT count(*) FROM retail.branches",
    "customers": "SELECT count(*) FROM retail.customers",
    "accounts": "SELECT count(*) FROM retail.accounts",
    "account_holders": "SELECT count(*) FROM retail.account_holders",
    "transactions": "SELECT count(*) FROM retail.transactions",
    "fx_rates_log": "SELECT count(*) FROM retail.fx_rates_log",
    "customers with NULL email": "SELECT count(*) FROM retail.customers WHERE email IS NULL",
    "customers whose notes contain a newline": r"SELECT count(*) FROM retail.customers WHERE notes LIKE E'%\n%'",
    "customers with empty tags array": "SELECT count(*) FROM retail.customers WHERE tags = '{}'",
    "accounts with negative balance": "SELECT count(*) FROM retail.accounts WHERE balance < 0",
    "fx duplicate (date, currency) groups": "SELECT count(*) FROM (SELECT 1 FROM retail.fx_rates_log GROUP BY rate_date, currency HAVING count(*) > 1) d",
}
for label, q in checks.items():
    print("%-42s %s" % (label, pg_query(q)[0][0]))

## 2. Spark session (used to register Hive tables and to validate)
Same configuration as your working notebook, plus the Postgres JDBC driver for the optional fallback.

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("Sqoop-POC")
    .config("spark.sql.catalogImplementation", "hive")
    .master("spark://spark-master:7077")
    #.master("local[*]")
    .config("spark.hadoop.fs.defaultFS", HDFS_URI)
    .config("spark.sql.warehouse.dir", HDFS_URI + "/user/hive/warehouse")
    .config("hive.metastore.uris", "thrift://172.16.0.4:9083")
    #.config("spark.sql.hive.metastore.version", "4.1.0")
    .config("spark.sql.hive.metastore.version", "3.1.3")
    .config("spark.sql.hive.metastore.jars", "maven")
    .config("spark.cores.max", "2")
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.3")
    .enableHiveSupport()
    .getOrCreate()
)
print("Spark", spark.version, "| master:", spark.sparkContext.master)

## 3. Sqoop: prerequisites and helpers
Checks the tools exist and stores the Postgres password in a protected HDFS file (so it is never on a command line or in a saved job).

In [ ]:
def sh(cmd, check=True, input_text=None, show_tail=12):
    """Run a command, keep the output, print a short tail (or everything on failure)."""
    print("$", " ".join(shlex.quote(c) for c in cmd))
    p = subprocess.Popen(cmd, stdin=subprocess.PIPE if input_text is not None else None,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, universal_newlines=True)
    out, _ = p.communicate(input_text)
    lines = out.splitlines()
    if p.returncode != 0:
        print("\n".join(lines[-80:]))
        if check:
            raise RuntimeError("command failed (exit %d): %s" % (p.returncode, cmd[0]))
    elif show_tail:
        print("\n".join(lines[-show_tail:]))
    return out


def hdfs(*args, **kw):
    return sh([HDFS_BIN, "dfs"] + list(args), **kw)


if USE_SQOOP:
    for tool in (SQOOP_BIN, HDFS_BIN):
        found = subprocess.call(["which", tool], stdout=subprocess.DEVNULL) == 0
        print("%-8s %s" % (tool, "found" if found else "NOT FOUND on PATH (set SQOOP_BIN/HDFS_BIN or use the Spark JDBC fallback in section 8)"))

    hdfs("-mkdir", "-p", RAW_BASE, "/user/%s" % HDFS_USER, show_tail=0)
    hdfs("-rm", "-f", PW_FILE, check=False, show_tail=0)
    sh([HDFS_BIN, "dfs", "-put", "-", PW_FILE], input_text=PG["password"], show_tail=0)   # no trailing newline
    hdfs("-chmod", "400", PW_FILE, show_tail=0)
    print("password file:", PW_FILE)
    # Reminder: the Postgres JDBC jar must be in $SQOOP_HOME/lib (e.g. postgresql-42.x.jar)

### Shared Sqoop options
* `--fields-terminated-by '\001'` (Hive's default delimiter) and `--null-string '\\N'`: so NULL stays NULL and is not confused with an empty string.
* `--hive-drop-import-delims`: strips `\n`, `\r`, `\001` from string columns. The `notes` and `memo` columns contain newlines and tabs on purpose; without this a text row would break into several lines.

In [ ]:
COMMON = [
    "--connect", JDBC_URL,
    "--username", PG["user"],
    "--password-file", PW_FILE,
    "--fields-terminated-by", "\\001",
    "--lines-terminated-by", "\\n",
    "--null-string", "\\\\N",
    "--null-non-string", "\\\\N",
    "--hive-drop-import-delims",
]
SCHEMA_ARGS = ["--", "--schema", PG_SCHEMA]      # Postgres connector argument: tables live in schema "retail"


def target(name):
    return "%s/%s" % (RAW_BASE, name)


def sqoop_import(name, extra, mappers):
    """Full (re)load: wipe the target dir, then run `sqoop import`."""
    hdfs("-rm", "-r", "-f", "-skipTrash", target(name), check=False, show_tail=0)
    cmd = [SQOOP_BIN, "import"] + COMMON + ["--target-dir", target(name), "-m", str(mappers)] + extra
    return sh(cmd)

## 4. Sqoop: full loads

| Job | Technique |
|---|---|
| `branches` | plain `--table`, 1 mapper |
| `customers` | **free-form `--query`** with casts: the enum (`::text`), JSONB (`::text`) and array (`array_to_string`) cannot be imported directly; 4 parallel mappers via `--split-by` |
| `account_holders` | composite PK, so `--split-by account_id` is required for parallel mappers |
| `fx_rates_log` | no PK and duplicates, so a single mapper (`--autoreset-to-one-mapper`) |
| `acct_enriched` | **free-form join** of accounts + customers + branches |

In [ ]:
CUSTOMERS_QUERY = """
SELECT c.customer_id, c.customer_ref, c.full_name, c.email,
       c.segment::text                    AS segment,
       c.country, c.kyc_status, c.pep_flag, c.date_of_birth, c.onboard_ts,
       c.address::text                    AS address_json,
       array_to_string(c.tags, '|')       AS tags,
       c.notes, c.updated_at
FROM retail.customers c
WHERE $CONDITIONS
"""

ENRICHED_QUERY = """
SELECT a.account_id, a.account_no, c.customer_ref, c.full_name,
       c.segment::text AS segment, a.account_type, a.currency, a.balance,
       a.branch_code, b.branch_name, b.city, a.status
FROM retail.accounts a
JOIN retail.customers c ON c.customer_id = a.customer_id
LEFT JOIN retail.branches b ON b.branch_code = a.branch_code
WHERE $CONDITIONS
"""

if USE_SQOOP:
    sqoop_import("branches", ["--table", "branches"] + SCHEMA_ARGS, 1)

    sqoop_import("customers", ["--query", CUSTOMERS_QUERY, "--split-by", "c.customer_id"], 4)

    sqoop_import("account_holders",
                 ["--table", "account_holders", "--split-by", "account_id"] + SCHEMA_ARGS, 4)

    sqoop_import("fx_rates_log",
                 ["--table", "fx_rates_log", "--autoreset-to-one-mapper"] + SCHEMA_ARGS, 1)

    sqoop_import("acct_enriched", ["--query", ENRICHED_QUERY, "--split-by", "a.account_id"], 4)

## 5. Sqoop: incremental loads as saved jobs
A saved job remembers the last value, so each run only moves new data.

* `transactions` : `--incremental append` on `txn_id` (new rows only, 8 parallel mappers).
* `accounts` : `--incremental lastmodified` on `updated_at` with `--merge-key account_id` (changed rows replace the old version, new rows are added).

In [ ]:
def create_saved_job(job_name, import_args):
    sh([SQOOP_BIN, "job", "--delete", job_name], check=False, show_tail=0)
    sh([SQOOP_BIN, "job", "--create", job_name, "--", "import"] + import_args, show_tail=3)


def exec_saved_job(job_name):
    return sh([SQOOP_BIN, "job", "--exec", job_name], show_tail=15)


TX_JOB = [
    "--table", "transactions", "--target-dir", target("transactions"),
    "--split-by", "txn_id", "-m", "8",
    "--incremental", "append", "--check-column", "txn_id", "--last-value", "0",
] + SCHEMA_ARGS

ACCT_JOB = [
    "--table", "accounts", "--target-dir", target("accounts"),
    "--split-by", "account_id", "-m", "4",
    "--incremental", "lastmodified", "--check-column", "updated_at",
    "--last-value", "1970-01-01 00:00:00", "--merge-key", "account_id",
] + SCHEMA_ARGS

if USE_SQOOP:
    hdfs("-rm", "-r", "-f", "-skipTrash", target("transactions"), target("accounts"), check=False, show_tail=0)
    create_saved_job("poc_transactions_append", COMMON + TX_JOB)
    create_saved_job("poc_accounts_lastmodified", COMMON + ACCT_JOB)
    exec_saved_job("poc_transactions_append")      # initial load: all 200,000 rows
    exec_saved_job("poc_accounts_lastmodified")    # initial load: all 15,000 rows

Look at what Sqoop landed (this is also what you will see in Hue's File Browser):

In [ ]:
if USE_SQOOP:
    hdfs("-ls", "-R", RAW_BASE, show_tail=60)
    hdfs("-du", "-h", RAW_BASE, show_tail=20)

## 6. Register Hive tables over the landed files, then write curated Parquet

* `sqoop_poc.stg_<table>`: **external** text tables, exactly the files Sqoop wrote (so the incremental loads show up automatically).
* `sqoop_poc.<table>`: curated **Parquet** copies. `transactions` is joined to accounts (adds `customer_id`, `currency`) and **partitioned by `txn_date`**.

In [ ]:
HIVE_COLS = {
    "branches": [("branch_code", "string"), ("branch_name", "string"), ("city", "string"),
                 ("state_code", "string"), ("opened_date", "date"), ("is_active", "boolean")],
    "customers": [("customer_id", "bigint"), ("customer_ref", "string"), ("full_name", "string"),
                  ("email", "string"), ("segment", "string"), ("country", "string"),
                  ("kyc_status", "string"), ("pep_flag", "boolean"), ("date_of_birth", "date"),
                  ("onboard_ts", "timestamp"), ("address_json", "string"), ("tags", "string"),
                  ("notes", "string"), ("updated_at", "timestamp")],
    "accounts": [("account_id", "bigint"), ("account_no", "string"), ("customer_id", "bigint"),
                 ("account_type", "string"), ("currency", "string"), ("branch_code", "string"),
                 ("balance", "decimal(18,2)"), ("credit_limit", "decimal(18,2)"), ("status", "string"),
                 ("open_date", "date"), ("close_date", "date"), ("updated_at", "timestamp")],
    "account_holders": [("account_id", "bigint"), ("customer_id", "bigint"), ("holder_role", "string"),
                        ("share_pct", "decimal(5,2)"), ("since_date", "date")],
    "transactions": [("txn_id", "bigint"), ("account_id", "bigint"), ("txn_ts", "timestamp"),
                     ("txn_type", "string"), ("amount", "decimal(18,2)"), ("status", "string"),
                     ("channel", "string"), ("counterparty", "string"), ("memo", "string"),
                     ("updated_at", "timestamp")],
    "fx_rates_log": [("rate_date", "date"), ("currency", "string"), ("usd_rate", "decimal(18,6)"),
                     ("source", "string"), ("loaded_at", "timestamp")],
    "acct_enriched": [("account_id", "bigint"), ("account_no", "string"), ("customer_ref", "string"),
                      ("full_name", "string"), ("segment", "string"), ("account_type", "string"),
                      ("currency", "string"), ("balance", "decimal(18,2)"), ("branch_code", "string"),
                      ("branch_name", "string"), ("city", "string"), ("status", "string")],
}


def register_staging():
    spark.sql("CREATE DATABASE IF NOT EXISTS %s" % HIVE_DB)
    for name, cols in HIVE_COLS.items():
        col_sql = ", ".join("%s %s" % c for c in cols)
        spark.sql("DROP TABLE IF EXISTS %s.stg_%s" % (HIVE_DB, name))
        spark.sql(
            r"""CREATE EXTERNAL TABLE %s.stg_%s (%s)
                ROW FORMAT DELIMITED FIELDS TERMINATED BY '\001' NULL DEFINED AS '\\N'
                STORED AS TEXTFILE
                LOCATION '%s'""" % (HIVE_DB, name, col_sql, target(name)))
        print("registered %s.stg_%s" % (HIVE_DB, name))


def refresh_curated():
    for name in HIVE_COLS:
        spark.catalog.refreshTable("%s.stg_%s" % (HIVE_DB, name))
    for name in HIVE_COLS:
        if name == "transactions":
            accts = spark.table("%s.stg_accounts" % HIVE_DB).select("account_id", "customer_id", "currency")
            tx = (spark.table("%s.stg_transactions" % HIVE_DB)
                  .join(accts, "account_id", "left")
                  .withColumn("txn_date", F.to_date("txn_ts")))
            (tx.write.mode("overwrite").format("parquet").partitionBy("txn_date")
               .saveAsTable("%s.transactions" % HIVE_DB))
        else:
            (spark.table("%s.stg_%s" % (HIVE_DB, name)).write.mode("overwrite").format("parquet")
             .saveAsTable("%s.%s" % (HIVE_DB, name)))
        print("curated %s.%s" % (HIVE_DB, name))


if USE_SQOOP:
    register_staging()
    refresh_curated()
    spark.sql("SHOW TABLES IN %s" % HIVE_DB).show(50, False)

## 7. Validate: Postgres vs Hive
Row counts, sums of money columns, and NULL counts must match. The text columns that held newlines are compared as "has newline" (Hive should have none, because they were stripped on import).

In [ ]:
def validate():
    pairs = [
        ("rows branches",          "SELECT count(*) FROM retail.branches",         "SELECT count(*) FROM %s.stg_branches" % HIVE_DB),
        ("rows customers",         "SELECT count(*) FROM retail.customers",        "SELECT count(*) FROM %s.stg_customers" % HIVE_DB),
        ("rows accounts",          "SELECT count(*) FROM retail.accounts",         "SELECT count(*) FROM %s.stg_accounts" % HIVE_DB),
        ("rows account_holders",   "SELECT count(*) FROM retail.account_holders",  "SELECT count(*) FROM %s.stg_account_holders" % HIVE_DB),
        ("rows transactions",      "SELECT count(*) FROM retail.transactions",     "SELECT count(*) FROM %s.stg_transactions" % HIVE_DB),
        ("rows transactions (curated)", "SELECT count(*) FROM retail.transactions", "SELECT count(*) FROM %s.transactions" % HIVE_DB),
        ("rows fx_rates_log (dups kept)", "SELECT count(*) FROM retail.fx_rates_log", "SELECT count(*) FROM %s.stg_fx_rates_log" % HIVE_DB),
        ("rows acct_enriched",     "SELECT count(*) FROM retail.accounts",         "SELECT count(*) FROM %s.stg_acct_enriched" % HIVE_DB),
        ("sum accounts.balance",   "SELECT sum(balance) FROM retail.accounts",     "SELECT sum(balance) FROM %s.stg_accounts" % HIVE_DB),
        ("sum transactions.amount", "SELECT sum(amount) FROM retail.transactions", "SELECT sum(amount) FROM %s.stg_transactions" % HIVE_DB),
        ("NULL customers.email",   "SELECT count(*) FROM retail.customers WHERE email IS NULL", "SELECT count(*) FROM %s.stg_customers WHERE email IS NULL" % HIVE_DB),
        ("NULL customers.date_of_birth", "SELECT count(*) FROM retail.customers WHERE date_of_birth IS NULL", "SELECT count(*) FROM %s.stg_customers WHERE date_of_birth IS NULL" % HIVE_DB),
        ("NULL transactions.channel", "SELECT count(*) FROM retail.transactions WHERE channel IS NULL", "SELECT count(*) FROM %s.stg_transactions WHERE channel IS NULL" % HIVE_DB),
        ("empty-string notes (not NULL)", "SELECT count(*) FROM retail.customers WHERE notes = ''", "SELECT count(*) FROM %s.stg_customers WHERE notes = ''" % HIVE_DB),
        ("PG notes with newline -> Hive 0", "SELECT 0", r"SELECT count(*) FROM %s.stg_customers WHERE notes LIKE '%%\n%%'" % HIVE_DB),
    ]
    ok = True
    print("%-34s %18s %18s  %s" % ("check", "postgres", "hive", ""))
    for label, pg_sql, hv_sql in pairs:
        a = pg_query(pg_sql)[0][0]
        b = spark.sql(hv_sql).collect()[0][0]
        same = (str(a) == str(b)) or (a is not None and b is not None and float(a) == float(b))
        ok &= bool(same)
        print("%-34s %18s %18s  %s" % (label, a, b, "OK" if same else "MISMATCH"))
    print("\nALL CHECKS PASSED" if ok else "\nSOME CHECKS FAILED")


if USE_SQOOP:
    validate()

In [ ]:
# Peek at the complex columns on the Hive side
if USE_SQOOP:
    spark.sql("""SELECT customer_ref, segment, tags, address_json, notes
                 FROM %s.customers WHERE tags IS NOT NULL LIMIT 5""" % HIVE_DB).show(5, False)
    spark.sql("""SELECT txn_date, count(*) AS txns, round(sum(amount), 2) AS total
                 FROM %s.transactions GROUP BY txn_date ORDER BY txn_date DESC LIMIT 5""" % HIVE_DB).show()

## 8. Simulate a day of changes and run the incremental jobs again
Adds 5,000 transactions, changes 300 accounts and adds 50 accounts in Postgres. Only that delta should move.

In [ ]:
if USE_SQOOP:
    before_tx = spark.sql("SELECT count(*) FROM %s.stg_transactions" % HIVE_DB).collect()[0][0]
    run_sql_file("03_incremental_changes.sql")
    exec_saved_job("poc_transactions_append")      # appends only txn_id > last value
    exec_saved_job("poc_accounts_lastmodified")    # merges rows with updated_at > last value
    refresh_curated()
    after_tx = spark.sql("SELECT count(*) FROM %s.stg_transactions" % HIVE_DB).collect()[0][0]
    print("transactions in Hive: %d -> %d (expected +5000)" % (before_tx, after_tx))
    validate()

## 9. Fallback: Spark JDBC instead of Sqoop
Use this if `sqoop` is not installed where the notebook runs. Set `USE_SQOOP = False` in the config cell, run sections 0-2, then run this section. It reads Postgres in parallel and writes Parquet tables into `sqoop_poc_spark`.

In [ ]:
RUN_JDBC_FALLBACK = not USE_SQOOP
JDBC_DB = "sqoop_poc_spark"

JDBC_SOURCES = {   # name -> (dbtable or subquery, partition column or None)
    "branches": ("retail.branches", None),
    "customers": ("""(SELECT c.customer_id, c.customer_ref, c.full_name, c.email, c.segment::text AS segment,
                            c.country, c.kyc_status, c.pep_flag, c.date_of_birth, c.onboard_ts,
                            c.address::text AS address_json, array_to_string(c.tags, '|') AS tags,
                            c.notes, c.updated_at FROM retail.customers c) t""", "customer_id"),
    "accounts": ("retail.accounts", "account_id"),
    "account_holders": ("retail.account_holders", "account_id"),
    "transactions": ("retail.transactions", "txn_id"),
    "fx_rates_log": ("retail.fx_rates_log", None),
}


def jdbc_copy(name, dbtable, part_col, partitions=8):
    reader = (spark.read.format("jdbc").option("url", JDBC_URL).option("dbtable", dbtable)
              .option("user", PG["user"]).option("password", PG["password"])
              .option("driver", "org.postgresql.Driver"))
    if part_col:
        lo, hi = pg_query("SELECT min(%s), max(%s) FROM %s" % (part_col, part_col,
                          "retail." + name))[0]
        reader = (reader.option("partitionColumn", part_col).option("lowerBound", int(lo))
                  .option("upperBound", int(hi)).option("numPartitions", partitions))
    df = reader.load()
    if name == "transactions":
        df = df.withColumn("txn_date", F.to_date("txn_ts"))
        df.write.mode("overwrite").format("parquet").partitionBy("txn_date").saveAsTable("%s.%s" % (JDBC_DB, name))
    else:
        df.write.mode("overwrite").format("parquet").saveAsTable("%s.%s" % (JDBC_DB, name))
    print("%-16s %d rows -> %s.%s" % (name, spark.table("%s.%s" % (JDBC_DB, name)).count(), JDBC_DB, name))


if RUN_JDBC_FALLBACK:
    spark.sql("CREATE DATABASE IF NOT EXISTS %s" % JDBC_DB)
    for n, (tbl, col) in JDBC_SOURCES.items():
        jdbc_copy(n, tbl, col)

## 10. See it in Hue
* **File Browser**: `/data/sqoop_poc/raw/<table>/part-m-*` are the raw Sqoop files.
* **Hive / SQL editor**: run
  ```sql
  SHOW TABLES IN sqoop_poc;
  SELECT segment, count(*) FROM sqoop_poc.customers GROUP BY segment;
  SELECT txn_date, count(*) FROM sqoop_poc.transactions GROUP BY txn_date ORDER BY txn_date DESC;
  ```

## Notes
* **Hive 4.1 metastore**: Sqoop's own `--hive-import` uses an old Hive client and often fails against Hive 4. That is why this notebook lets Sqoop write files to HDFS and registers the tables with Spark instead.
* **Re-running**: the notebook drops and recreates the Postgres schema, wipes the HDFS target dirs and replaces the Hive tables, so it can be run again from the top.
* **Cleanup**: `DROP DATABASE sqoop_poc CASCADE;` in Hue, and `hdfs dfs -rm -r /data/sqoop_poc` for the raw files.

In [ ]:
spark.sql("SHOW DATABASES").show(truncate=False)